# Symbolic Knowledge Discovery - XWine Dataset (raw data)

This notebook follows the workflow style from:
- `resources/FCA_From_Scratch_SymbolicKD.ipynb`
- `resources/SymbolicKD_Pattern_Mining_+_Clustering.ipynb`

The run is intentionally lightweight to avoid computationally heavy processing:
- full binary-context mining with `caspailleur`
- sampled pattern-space mining with `paspailleur` for subgroup exploration


In [ ]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = Path("outputs/xwine_symbolic_kd_raw_light")
summary = json.loads((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8"))
summary


## 1) Context and Dataset Overview

We mine a binary context for FCA (caspailleur) and a compact multi-dimensional pattern structure for subgroup-oriented exploration (paspailleur).


In [ ]:
context = pd.read_csv(OUTPUT_DIR / "binary_context.csv")
concepts = pd.read_csv(OUTPUT_DIR / "caspailleur_concepts.csv")
implications = pd.read_csv(OUTPUT_DIR / "caspailleur_implications.csv")
feature_importance = pd.read_csv(OUTPUT_DIR / "feature_importance.csv")
interesting_implications = pd.read_csv(OUTPUT_DIR / "interesting_implications.csv")
paspailleur_concepts = pd.read_csv(OUTPUT_DIR / "paspailleur_concepts.csv")

print("Context shape:", context.shape)
print("Caspailleur concepts:", len(concepts))
print("Caspailleur implications:", len(implications))
print("Paspailleur concepts (sampled pattern structure):", len(paspailleur_concepts))


## 2) Feature Importance (more important vs less important)

Importance combines:
- binary support (frequency)
- stability-weighted concept participation
- implication participation weight


In [ ]:
top_features = feature_importance.head(15).copy()
low_features = feature_importance.tail(15).copy()

display(top_features)
display(low_features)

plt.figure(figsize=(10, 5))
plt.barh(top_features["feature"][::-1], top_features["importance_score"][::-1])
plt.title("Top 15 features by symbolic importance score")
plt.xlabel("importance_score")
plt.tight_layout()
plt.show()


## 3) Concepts (including cool/stable concepts)

`caspailleur` returns concept intents/extents with support and delta stability.
We inspect high-support and high-stability concepts.


In [ ]:
stable_concepts = concepts.sort_values(["delta_stability", "support"], ascending=[False, False]).head(20)
support_concepts = concepts.sort_values("support", ascending=False).head(20)

display(stable_concepts)
display(support_concepts)


## 4) Implications (highlight interesting ones)

The table below shows a compact ranked set of implications to discuss (3-5 most interesting).


In [ ]:
display(interesting_implications.head(5))

if len(interesting_implications) > 0:
    plt.figure(figsize=(8, 4))
    plt.bar(range(len(interesting_implications.head(5))), interesting_implications.head(5)["interesting_score"])
    plt.title("Interesting implication scores (top 5)")
    plt.xlabel("Implication rank")
    plt.ylabel("interesting_score")
    plt.tight_layout()
    plt.show()


## 5) Subgroups (binary targets + rationale)

Subgroups are mined as patterns that are highly precise for a **binary target**.

Detected subgroup files: `subgroups_high_abv_quartile.csv`.

Rationale used in this run:
- `high_abv_quartile`: ABV top quartile (>= 14.15) isolates stronger wine profiles.

Temporal-grouping policy:
- Prefer broad eras (e.g. <=2010 vs >2010) rather than tiny year-pairs (2001-2002, 2002-2003, ...)
- Reason: broader bins reduce sparsity and improve subgroup interpretability/stability


In [ ]:
for csv_file in sorted(OUTPUT_DIR.glob("subgroups_*.csv")):
    print("\n===", csv_file.name, "===")
    df = pd.read_csv(csv_file)
    display(df.head(15))


## 6) Paspailleur Pattern Concepts (sampled run)

This section mirrors the pattern-mining spirit from the resource notebook but under lightweight constraints.


In [ ]:
display(paspailleur_concepts.head(20))


## 7) Practical Interpretation Notes

- Use top symbolic features as candidate explanatory variables for recommendation/ranking models.
- Validate highlighted implications with domain experts (oenologists/sommeliers).
- Subgroups should be re-mined on larger compute budgets if exhaustive subgroup quality optimization is required.
